# Iter Sibiricum: первые шаги с данными

Тетрадка читает открытые файлы сайта [iter-sibiricum.ru](https://iter-sibiricum.ru) напрямую по адресу — ничего скачивать не нужно. Подробное описание файлов: раздел [«Данные»](https://iter-sibiricum.ru/#data).

*Getting started with the Iter Sibiricum data: the notebook reads the site's open files directly.*

In [ ]:
import pandas as pd, requests, json
BASE = "https://iter-sibiricum.ru/"
get = lambda path: requests.get(BASE + path, timeout=60).json()

works = pd.DataFrame(get("data/works.json"))
groups = get("data/groups.json")
print(len(works), "книг;", len(groups), "групп")
works[["id", "author", "title_short", "year", "nat", "period", "stay", "words_sib", "per10k"]].head()

## 1. Сколько пишут о народах в разные эпохи
`per10k` — упоминаний народов на 10 тыс. слов текста о Сибири.

In [ ]:
works.groupby("period").agg(книг=("id", "size"), слов_о_Сибири=("words_sib", "sum"), упоминаний_на_10к=("per10k", "mean")).round(1)

## 2. Свидетельства: кто чаще писал о бурятах
Файл `data/evidence/g/<группа>.json` — все абзацы, где упомянута группа: книга, номер абзаца, о Сибири ли абзац, фрагмент текста, позиции упоминаний, места в том же абзаце.

In [ ]:
ev = get("data/evidence/g/buryat.json")["r"]
rows = pd.DataFrame(ev, columns=["work", "paragraph", "about_siberia", "fragment", "marks", "places"])
rows = rows.merge(works[["id", "author", "nat", "period", "words_sib"]], left_on="work", right_on="id")
# абзацы о бурятах на 10 тыс. слов о Сибири — по стране автора
by = rows.groupby("nat").size().to_frame("абзацев")
by["слов_о_Сибири"] = works.groupby("nat")["words_sib"].sum()
by["на_10к_слов"] = (by["абзацев"] / by["слов_о_Сибири"] * 1e4).round(2)
by.sort_values("на_10к_слов", ascending=False)

In [ ]:
# ссылка на каждый абзац в читалке сайта
rows["url"] = BASE + "#book-" + rows["work"] + "-p" + rows["paragraph"].astype(str).str.zfill(5)
rows[["author", "period", "fragment", "url"]].head(5)

## 3. Маршруты на карте
`data/export/iter-sibiricum-routes.geojson` открывается в geopandas и QGIS.

In [ ]:
import geopandas as gpd
routes = gpd.read_file(BASE + "data/export/iter-sibiricum-routes.geojson")
print(routes.shape)
routes.head(3)

In [ ]:
ax = routes.plot(figsize=(12, 6), linewidth=0.8, alpha=0.6)
ax.set_xlim(20, 180); ax.set_ylim(35, 78); ax.set_title("Маршруты Iter Sibiricum")

## Как цитировать
Iter Sibiricum: европейские травелоги о Сибири, 1800–1930 : цифровой ресурс. — URL: https://iter-sibiricum.ru. Наборы данных на Zenodo: [10.5281/zenodo.24531924](https://doi.org/10.5281/zenodo.24531924), [10.5281/zenodo.24532195](https://doi.org/10.5281/zenodo.24532195).